In [6]:
import pandas as pd

pd.set_option('display.max_columns', 30) # Указываем максимальную ширину дисплея для просмотра всех столбцов
melb_df = pd.read_csv('data/melb_data_fe.csv')
melb_df.head(2)

,Suburb,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,Bedroom,Bathroom,Car,Landsize,BuildingArea,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount,MeanRoomsSquare,AreaRatio,MonthSale,AgeBuilding,WeekdaySale,StreetType,Weekend
0,Abbotsford,2,house,1480000.0,S,Biggin,2016-03-12,2.5,3067,2,1,1,202.0,126.0,Yarra,-37.7996,144.9984,Northern Metropolitan,4019,25.2,-0.231707,3,46,5,St,1
1,Abbotsford,2,house,1035000.0,S,Biggin,2016-04-02,2.5,3067,2,1,0,156.0,79.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019,15.8,-0.327660,4,116,5,St,1


In [7]:
melb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 26 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Suburb           13580 non-null  object 
 1   Rooms            13580 non-null  int64  
 2   Type             13580 non-null  object 
 3   Price            13580 non-null  float64
 4   Method           13580 non-null  object 
 5   SellerG          13580 non-null  object 
 6   Date             13580 non-null  object 
 7   Distance         13580 non-null  float64
 8   Postcode         13580 non-null  int64  
 9   Bedroom          13580 non-null  int64  
 10  Bathroom         13580 non-null  int64  
 11  Car              13580 non-null  int64  
 12  Landsize         13580 non-null  float64
 13  BuildingArea     13580 non-null  float64
 14  CouncilArea      12211 non-null  object 
 15  Lattitude        13580 non-null  float64
 16  Longtitude       13580 non-null  float64
 17  Regionname  

In [8]:
melb_qv = pd.to_datetime(melb_df['Date']).dt.quarter.value_counts()
melb_qv

Date
3    4873
2    4359
4    2329
1    2019
Name: count, dtype: int64

In [9]:
exclude_list = ['Date', 'Rooms', 'Bedroom', 'Bathroom', 'Car']
for col in melb_df.columns:
    if melb_df[col].nunique() < 150 and col not in exclude_list:
        melb_df[col] = melb_df[col].astype('category')

In [10]:
melb_df.sort_values(by= ['Price'])[['Price','Rooms','Type','Suburb']]

,Price,Rooms,Type,Suburb
2652,85000.0,1,unit,Footscray
1805,131000.0,4,house,other
7303,145000.0,1,unit,Albion
1927,145000.0,4,house,Coburg
7940,160000.0,1,unit,Hawthorn
...,...,...,...,...
12557,6400000.0,5,house,Middle Park
3616,6500000.0,6,house,Kew
9575,7650000.0,4,house,Hawthorn
7692,8000000.0,5,house,Canterbury


In [11]:
melb_df[melb_df['Bathroom'] >= 2].sort_values(by='Price')[['Price', 'Rooms','Bathroom', 'Type', 'Suburb']].head(3)

,Price,Rooms,Bathroom,Type,Suburb
8871,320000.0,4,2,house,other
5826,320000.0,1,2,unit,St Kilda
3432,320000.0,3,2,house,other


by — имя или список имён столбцов, по значениям которых производится сортировка.

axis — ось, по которой производится сортировка (0 — строки, 1 — столбцы). По умолчанию сортировка производится по строкам.

ascending — сортировка по возрастанию (от меньшего к большему). По умолчанию параметр выставлен на True, для сортировки по убыванию (от большего к меньшему) необходимо выставить его на False.

ignore_index — создаются ли новые индексы в таблице. По умолчанию выставлен на False и сохраняет индексы изначальной таблицы.

inplace — производится ли замена исходной таблицы на отсортированную. По умолчанию параметр выставлен на False, то есть замены не производится. Чтобы переопределить исходную таблицу на отсортированную, необходимо выставить этот параметр на True.

In [12]:
mask1 = melb_df['AreaRatio'] < -0.8
mask2 = melb_df['Type'] == 'townhouse'
mask3 = melb_df['SellerG'] == 'McGrath'
melb_df[mask1 & mask2 & mask3].sort_values(by=['Date', 'AreaRatio'], ascending=[True, False], ignore_index=True)[['Date', 'AreaRatio']]

,Date,AreaRatio
0,2016-07-26,-0.974922
1,2016-09-24,-0.971831
2,2016-11-27,-0.953608
3,2016-12-11,-0.945946
4,2017-08-04,-0.947368
5,2017-08-04,-0.970874


In [13]:
mask1 = (melb_df['Type'] == 'townhouse') & (melb_df['Rooms'] > 2)

melb_df[mask1].sort_values(by=['Rooms', 'MeanRoomsSquare'], ascending=[True, False], ignore_index=True).loc[18,['Price']] # type: ignore

Price    1300000.0
Name: 18, dtype: object

In [14]:
melb_df['Type'].value_counts()

Type
house        9449
unit         3017
townhouse    1114
Name: count, dtype: int64

In [15]:
round(melb_df.groupby(by='Type', observed=True)['Price'].mean(),2)

Type
house        1242664.76
townhouse     933735.05
unit          605127.48
Name: Price, dtype: float64

In [16]:
melb_df.groupby('Regionname', observed=True)['Distance'].min().sort_values(ascending=False)

Regionname
Western Victoria              29.8
Eastern Victoria              25.2
Northern Victoria             21.8
South-Eastern Metropolitan    14.7
Eastern Metropolitan           7.8
Western Metropolitan           4.3
Southern Metropolitan          0.7
Northern Metropolitan          0.0
Name: Distance, dtype: float64

In [17]:
melb_df.groupby('MonthSale', as_index=False, observed=True)['Price'].agg(['count','mean','max']).sort_values(by='count', ascending=False, ignore_index=True)

,MonthSale,count,mean,max
0,8,1850,1.056371e+06,6500000.0
1,7,1835,9.314698e+05,9000000.0
2,5,1644,1.097807e+06,8000000.0
3,6,1469,1.068981e+06,7650000.0
4,3,1408,1.146762e+06,5600000.0
5,4,1246,1.050479e+06,5500000.0
6,9,1188,1.126349e+06,6400000.0
7,10,854,1.135970e+06,6250000.0
8,11,750,1.142503e+06,5050000.0
9,12,725,1.144737e+06,5700000.0


In [18]:
melb_df.groupby('MonthSale', observed=True)['Price'].agg('describe')

,count,mean,std,min,25%,50%,75%,max
MonthSale,,,,,,,,
1,278.0,9.397921e+05,577668.924214,170000.0,570500.0,795000.0,1111250.0,5200000.0
2,333.0,1.169051e+06,671564.357417,131000.0,710000.0,1020000.0,1478000.0,4735000.0
3,1408.0,1.146762e+06,709573.596867,85000.0,680000.0,945000.0,1400000.0,5600000.0
4,1246.0,1.050479e+06,591892.902979,145000.0,655000.0,905500.0,1298750.0,5500000.0
5,1644.0,1.097807e+06,668492.867996,145000.0,650000.0,905000.0,1371250.0,8000000.0
6,1469.0,1.068981e+06,606010.069052,222000.0,660000.0,900000.0,1325000.0,7650000.0
7,1835.0,9.314698e+05,537390.803161,190000.0,586750.0,800000.0,1150000.0,9000000.0
8,1850.0,1.056371e+06,619617.476541,160000.0,635000.0,892000.0,1310000.0,6500000.0
9,1188.0,1.126349e+06,608734.690742,170000.0,725000.0,980000.0,1360000.0,6400000.0


In [19]:
melb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 26 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   Suburb           13580 non-null  category
 1   Rooms            13580 non-null  int64   
 2   Type             13580 non-null  category
 3   Price            13580 non-null  float64 
 4   Method           13580 non-null  category
 5   SellerG          13580 non-null  category
 6   Date             13580 non-null  object  
 7   Distance         13580 non-null  float64 
 8   Postcode         13580 non-null  int64   
 9   Bedroom          13580 non-null  int64   
 10  Bathroom         13580 non-null  int64   
 11  Car              13580 non-null  int64   
 12  Landsize         13580 non-null  float64 
 13  BuildingArea     13580 non-null  float64 
 14  CouncilArea      12211 non-null  category
 15  Lattitude        13580 non-null  float64 
 16  Longtitude       13580 non-null  float64

In [20]:
melb_df.groupby('Regionname', observed=True)['SellerG'].agg(['nunique', set]).sort_values(by='nunique', ascending=False)

,nunique,set
Regionname,,
Northern Metropolitan,40,"{McGrath, Buckingham, Kay, Village, Sweeney, R..."
Southern Metropolitan,38,"{Buxton, McGrath, Buckingham, Kay, RT, Raine, ..."
Western Metropolitan,34,"{McGrath, Village, Sweeney, RT, Raine, Greg, B..."
Eastern Metropolitan,26,"{McGrath, Buckingham, Buxton, Kay, RT, Fletche..."
South-Eastern Metropolitan,25,"{Buxton, McGrath, Greg, Fletchers, C21, Eview,..."
Eastern Victoria,11,"{C21, McGrath, Harcourts, Eview, hockingstuart..."
Northern Victoria,11,"{LITTLE, McGrath, Buckingham, hockingstuart, M..."
Western Victoria,6,"{hockingstuart, other, HAR, Raine, Ray, YPA}"


In [21]:
melb_df.groupby('Regionname', observed=True)['Lattitude'].agg('std').sort_values(ascending=True)

Regionname
Western Victoria              0.011579
Southern Metropolitan         0.043080
Eastern Metropolitan          0.047890
Northern Metropolitan         0.049639
Western Metropolitan          0.051251
South-Eastern Metropolitan    0.073411
Northern Victoria             0.084455
Eastern Victoria              0.147067
Name: Lattitude, dtype: float64

In [22]:
melb_df['Date'] = pd.to_datetime(melb_df['Date'])
mask = (melb_df['Date'] >= '2017-05-01') & (melb_df['Date'] <= '2017-09-01')
melb_df[mask].groupby('SellerG', observed=True)['Price'].sum().sort_values(ascending=True).head(1)

SellerG
LITTLE    2742000.0
Name: Price, dtype: float64

In [23]:
round(melb_df.groupby('Rooms', observed=True)['Price'].mean().sort_values(ascending=False),1)

Rooms
7     1920700.0
5     1870260.4
6     1849365.7
8     1602750.0
4     1445281.7
3     1076080.6
10     900000.0
2      775081.2
1      433824.5
Name: Price, dtype: float64

In [24]:
melb_df.groupby('Rooms')[['Price', 'BuildingArea']].median()

,Price,BuildingArea
Rooms,,
1,385000.0,107.0
2,690000.0,126.0
3,950000.0,126.0
4,1285000.0,142.0
5,1660000.0,176.0
6,1800000.0,126.0
7,1496000.0,216.5
8,1515000.0,126.0
10,900000.0,126.0


In [25]:
import numpy as np

a = [1,2,3,4]
med = np.median(a)
print(med)

2.5


In [26]:
melb_df.groupby(['Rooms','Type'], observed=True)['Price'].median().unstack()

Type,house,townhouse,unit
Rooms,,,
1,857500.0,580000.0,377500.0
2,950000.0,675000.0,585000.0
3,990000.0,900000.0,780000.0
4,1300000.0,1157500.0,830000.0
5,1670000.0,1000000.0,NaN
6,1820000.0,NaN,520000.0
7,1496000.0,NaN,NaN
8,1150000.0,NaN,2250000.0
10,900000.0,NaN,NaN


Основные параметры метода pivot_table()

* values — имя столбца, по которому необходимо получить сводные данные, применяя агрегирующую функцию;
* index — имя столбца, значения которого станут строками сводной таблицы;
* columns — имя столбца, значения которого станут столбцами сводной таблицы;
* aggfunc — имя или список имён агрегирующих функций (по умолчанию — подсчёт среднего, 'mean');
* fill_value — значение, которым необходимо заполнить пропуски (по умолчанию пропуски не заполняются).

In [27]:

melb_df.pivot_table(values='Price', index='Rooms', columns='Type', fill_value=0, observed=False).round()

Type,house,townhouse,unit
Rooms,,,
1,866866.0,592705.0,389929.0
2,1017238.0,710158.0,610491.0
3,1109233.0,984709.0,850596.0
4,1462283.0,1217092.0,1037476.0
5,1877327.0,1035000.0,0.0
6,1869508.0,0.0,520000.0
7,1920700.0,0.0,0.0
8,1510286.0,0.0,2250000.0
10,900000.0,0.0,0.0


In [28]:
melb_df.pivot_table(values='Price', index='Regionname', columns='Weekend', aggfunc='count', observed=False)

Weekend,0,1
Regionname,,
Eastern Metropolitan,447,1024
Eastern Victoria,13,40
Northern Metropolitan,1258,2632
Northern Victoria,11,30
South-Eastern Metropolitan,123,327
Southern Metropolitan,1534,3161
Western Metropolitan,960,1988
Western Victoria,8,24


In [29]:
melb_df.pivot_table(values='Landsize', index='Regionname', columns='Type', aggfunc=['median', 'mean'], fill_value=0, observed=False)

median                          mean              \
Type                        house townhouse   unit        house   townhouse   
Regionname                                                                    
Eastern Metropolitan        674.0     233.5  203.0   717.422847  269.440678   
Eastern Victoria            843.0       0.0  230.0  3108.960000    0.000000   
Northern Metropolitan       459.5     134.0    0.0   619.249092  317.325733   
Northern Victoria           724.0       0.0    0.0  3355.463415    0.000000   
South-Eastern Metropolitan  630.5     240.0  199.0   664.306701  212.160000   
Southern Metropolitan       586.0     246.0    0.0   569.643881  278.858824   
Western Metropolitan        531.0     198.0   62.0   507.883406  244.560669   
Western Victoria            599.5       0.0    0.0   655.500000    0.000000   

                                        
Type                              unit  
Regionname                              
Eastern Metropolitan        330.444444  
Eastern Victoria            295.333333  
Northern Metropolitan       495.026538  
Northern Victoria             0.000000  
South-Eastern Metropolitan  357.864865  
Southern Metropolitan       466.380245  
Western Metropolitan        557.637232  
Western Victoria              0.000000

In [30]:
melb_df.pivot_table(values='Price', index=['Method', 'Type'], columns='Regionname', aggfunc=['median', 'mean'], fill_value=0, observed=False)

median                                         \
Regionname       Eastern Metropolitan Eastern Victoria Northern Metropolitan   
Method Type                                                                    
PI     house                1244000.0         780000.0              900000.0   
       townhouse             760000.0              0.0              632500.0   
       unit                  650000.0              0.0              410000.0   
S      house                1127000.0         675000.0              920000.0   
       townhouse             828000.0              0.0              750000.0   
       unit                  645750.0         492000.0              525500.0   
SA     house                 932500.0         950000.0              817500.0   
       townhouse             807500.0              0.0              425000.0   
       unit                       0.0              0.0              616000.0   
SP     house                1050000.0         672500.0              900000.0   
       townhouse             910000.0              0.0              690000.0   
       unit                  515000.0         400000.0              470000.0   
VB     house                1100000.0         712500.0             1050000.0   
       townhouse             892500.0              0.0              640000.0   
       unit                  500000.0              0.0              450000.0   

                                                               \
Regionname       Northern Victoria South-Eastern Metropolitan   
Method Type                                                     
PI     house              500000.0                   865000.0   
       townhouse               0.0                  1190000.0   
       unit                    0.0                   525000.0   
S      house              555000.0                   883300.0   
       townhouse               0.0                   875000.0   
       unit                    0.0                   606000.0   
SA     house              540000.0                   880000.0   
       townhouse               0.0                        0.0   
       unit                    0.0                        0.0   
SP     house              521000.0                   770000.0   
       townhouse               0.0                   800000.0   
       unit                    0.0                   601000.0   
VB     house              690000.0                   850000.0   
       townhouse               0.0                        0.0   
       unit                    0.0                   700000.0   

                                                                              \
Regionname       Southern Metropolitan Western Metropolitan Western Victoria   
Method Type                                                                    
PI     house                 1725000.0             870000.0         630000.0   
       townhouse             1055000.0             670000.0              0.0   
       unit                   571250.0             360000.0              0.0   
S      house                 1611000.0             870000.0         397500.0   
       townhouse             1135000.0             729000.0              0.0   
       unit                   655000.0             489000.0              0.0   
SA     house                 1390000.0             772500.0              0.0   
       townhouse             1141000.0             467500.0              0.0   
       unit                   580000.0             571000.0              0.0   
SP     house                 1521750.0             865000.0         360000.0   
       townhouse             1162500.0             702500.0              0.0   
       unit                   550000.0             460000.0              0.0   
VB     house                 1800000.0             880000.0              0.0   
       townhouse             1250000.0             689500.0              0.0   
       unit                   500000.0             420000.0

In [31]:
pivot = melb_df.pivot_table(values='Landsize', index='Regionname', columns='Type', aggfunc=['mean', 'median'], fill_value=0, observed=False)
pivot

mean                         median  \
Type                              house   townhouse        unit  house   
Regionname                                                               
Eastern Metropolitan         717.422847  269.440678  330.444444  674.0   
Eastern Victoria            3108.960000    0.000000  295.333333  843.0   
Northern Metropolitan        619.249092  317.325733  495.026538  459.5   
Northern Victoria           3355.463415    0.000000    0.000000  724.0   
South-Eastern Metropolitan   664.306701  212.160000  357.864865  630.5   
Southern Metropolitan        569.643881  278.858824  466.380245  586.0   
Western Metropolitan         507.883406  244.560669  557.637232  531.0   
Western Victoria             655.500000    0.000000    0.000000  599.5   

                                             
Type                       townhouse   unit  
Regionname                                   
Eastern Metropolitan           233.5  203.0  
Eastern Victoria                 0.0  230.0  
Northern Metropolitan          134.0    0.0  
Northern Victoria                0.0    0.0  
South-Eastern Metropolitan     240.0  199.0  
Southern Metropolitan          246.0    0.0  
Western Metropolitan           198.0   62.0  
Western Victoria                 0.0    0.0

In [32]:
pivot.columns # = ['_'.join(col).strip() for col in pivot.columns.values]

MultiIndex([(  'mean',     'house'),
            (  'mean', 'townhouse'),
            (  'mean',      'unit'),
            ('median',     'house'),
            ('median', 'townhouse'),
            ('median',      'unit')],
           names=[None, 'Type'])

In [33]:
pivot2 = ['_'.join(col).strip() for col in pivot.columns.values]
pivot2

['mean_house',
 'mean_townhouse',
 'mean_unit',
 'median_house',
 'median_townhouse',
 'median_unit']

In [34]:
pivot['median']['unit']

Regionname
Eastern Metropolitan          203.0
Eastern Victoria              230.0
Northern Metropolitan           0.0
Northern Victoria               0.0
South-Eastern Metropolitan    199.0
Southern Metropolitan           0.0
Western Metropolitan           62.0
Western Victoria                0.0
Name: unit, dtype: float64

In [35]:
pivot

mean                         median  \
Type                              house   townhouse        unit  house   
Regionname                                                               
Eastern Metropolitan         717.422847  269.440678  330.444444  674.0   
Eastern Victoria            3108.960000    0.000000  295.333333  843.0   
Northern Metropolitan        619.249092  317.325733  495.026538  459.5   
Northern Victoria           3355.463415    0.000000    0.000000  724.0   
South-Eastern Metropolitan   664.306701  212.160000  357.864865  630.5   
Southern Metropolitan        569.643881  278.858824  466.380245  586.0   
Western Metropolitan         507.883406  244.560669  557.637232  531.0   
Western Victoria             655.500000    0.000000    0.000000  599.5   

                                             
Type                       townhouse   unit  
Regionname                                   
Eastern Metropolitan           233.5  203.0  
Eastern Victoria                 0.0  230.0  
Northern Metropolitan          134.0    0.0  
Northern Victoria                0.0    0.0  
South-Eastern Metropolitan     240.0  199.0  
Southern Metropolitan          246.0    0.0  
Western Metropolitan           198.0   62.0  
Western Victoria                 0.0    0.0

In [36]:
mask = pivot['mean']['house'] < pivot['median']['house']
pivot[mask]

mean                         median            \
Type                        house   townhouse        unit  house townhouse   
Regionname                                                                   
Southern Metropolitan  569.643881  278.858824  466.380245  586.0     246.0   
Western Metropolitan   507.883406  244.560669  557.637232  531.0     198.0   

                             
Type                   unit  
Regionname                   
Southern Metropolitan   0.0  
Western Metropolitan   62.0

In [37]:
mult1 = melb_df.pivot_table(values='BuildingArea', index='Rooms', columns='Type', aggfunc='median', fill_value=0, observed=False)
mult1

Type,house,townhouse,unit
Rooms,,,
1,126.0,88.0,69.5
2,126.0,114.0,110.0
3,126.0,126.0,126.0
4,141.0,159.5,126.0
5,177.0,152.0,0.0
6,126.0,0.0,171.0
7,216.5,0.0,0.0
8,126.0,0.0,126.0
10,126.0,0.0,0.0


In [38]:
melb_df.pivot_table(values='Price', index='SellerG', columns='Type', aggfunc='median', fill_value=0, observed=False).sort_values(by='unit', ascending=False)

Type,house,townhouse,unit
SellerG,,,
Nick,2025000.0,780000.0,900000.0
Marshall,1975000.0,1408500.0,715000.0
Cayzer,1505000.0,1450000.0,707500.0
Kay,2220000.0,1365000.0,695000.0
Noel,1400500.0,990000.0,693250.0
Buxton,1323750.0,1030000.0,670000.0
Fletchers,1390000.0,1238000.0,653000.0
Chisholm,1520000.0,950000.0,640000.0
Philip,1035000.0,701000.0,636000.0


In [39]:
ratings1 = pd.read_csv('data/ratings1.csv')
ratings2 = pd.read_csv('data/ratings2.csv')
dates = pd.read_csv('data/dates.csv')
movies = pd.read_csv('data/movies.csv')

In [40]:
ratings1

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0
...,...,...,...
39996,274,5582,2.5
39997,274,5585,3.5
39998,274,5588,3.5
39999,274,5618,4.0


In [41]:
movies

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy
9739,193585,Flint (2017),Drama
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation


In [42]:
movies.nunique()

movieId    9742
title      9737
genres      951
dtype: int64

In [43]:
ratings1.nunique()

userId      274
movieId    6219
rating       10
dtype: int64

In [44]:
dates.value_counts().sort_values(ascending=False)

date               
2016-04-04 16:39:58    128
2016-04-04 16:39:57    124
2016-04-04 16:39:56     85
1996-03-29 18:36:56     37
2016-04-04 16:39:55     37
                      ... 
2018-09-18 01:49:36      1
2018-09-18 01:49:54      1
2018-09-18 17:58:03      1
2018-09-18 17:58:20      1
2018-09-17 04:27:31      1
Name: count, Length: 85043, dtype: int64

In [45]:
year_count = pd.to_datetime(dates['date']).dt.year.value_counts().nlargest()
year_count

date
2000    10061
2017     8198
2007     7114
2016     6703
2015     6616
Name: count, dtype: int64

In [46]:
ratings = pd.concat([ratings1, ratings2], ignore_index=True)
ratings

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0
...,...,...,...
100832,610,166534,4.0
100833,610,168248,5.0
100834,610,168250,5.0
100835,610,168252,5.0


In [47]:
print('Число строк в таблице ratings: ', ratings.shape[0])
print('Число строк в таблице dates: ', dates.shape[0])
print(ratings.shape[0] == dates.shape[0])

Число строк в таблице ratings:  100837
Число строк в таблице dates:  100836
False


In [48]:
display(ratings1.tail(1), ratings2.head(1))

,userId,movieId,rating
40000,274,5621,2.0


,userId,movieId,rating
0,274,5621,2.0


In [49]:
ratings.drop_duplicates(inplace=True, ignore_index=True)
ratings

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0
...,...,...,...
100831,610,166534,4.0
100832,610,168248,5.0
100833,610,168250,5.0
100834,610,168252,5.0


In [50]:
dates

,date
0,2000-07-30 18:45:03
1,2000-07-30 18:20:47
2,2000-07-30 18:37:04
3,2000-07-30 19:03:35
4,2000-07-30 18:48:51
...,...
100831,2017-05-03 21:53:22
100832,2017-05-03 22:21:31
100833,2017-05-08 19:50:47
100834,2017-05-03 21:19:12


In [51]:
print('Число строк в таблице ratings: ', ratings.shape[0])
print('Число строк в таблице dates: ', dates.shape[0])
print(ratings.shape[0] == dates.shape[0])

Число строк в таблице ratings:  100836
Число строк в таблице dates:  100836
True


In [52]:
ratings_dates = pd.concat([ratings, dates], axis=1)

In [53]:
ratings_dates.head(3)

,userId,movieId,rating,date
0,1,1,4.0,2000-07-30 18:45:03
1,1,3,4.0,2000-07-30 18:20:47
2,1,6,4.0,2000-07-30 18:37:04


# Описание задачи

В ваше распоряжение предоставлена директория `users` (`./Root/users`).  
В данной директории содержатся CSV-файлы, в каждом из которых хранится информация об идентификаторах пользователей (`user_id`) и ссылках на их фотографии (`image_url`).  
**Количество файлов в директории может быть любым.**

---

## Требуемая функция

Необходимо написать функцию `concat_user_files(path)`, где:

- **`path`** — путь до директории.

---

### Функция должна выполнить следующие действия:

1. **Объединить информацию** из всех CSV-файлов в единый `DataFrame`.
2. **Удалить дубликаты**.
3. **Обновить индексы** результирующей таблицы.
4. **Отсортировать пользователей** по числовой части `user_id` (игнорируя буквенную часть).

---

### Пример результата

Если в директории `users` есть несколько CSV-файлов, то результирующая таблица должна выглядеть так:

| index | user_id | image_url              |
|-------|---------|------------------------|
| 0     | user1   | https://example.com/1  |
| 1     | user2   | https://example.com/2  |
| 2     | user3   | https://example.com/3  |
| ...   | ...     | ...                    |

In [54]:
def concat_user_files2(path):
    df = pd.concat(
        [pd.read_csv(file) for file in path],
        ignore_index=True
    )

    df = df.drop_duplicates()

    # предполагаем формат user_id: 'user123'
    df['num'] = df['user_id'].apply(lambda x: int(x[4:]))

    df = df.sort_values('num').reset_index(drop=True)

    return df.drop(columns='num')

In [55]:
import os
import pandas as pd

def concat_user_files(path):
    df = pd.concat(
        [pd.read_csv(path + '/' + f) for f in os.listdir(path) if f.endswith('.csv')],
        ignore_index=True
    )

    df = df.drop_duplicates()
    df['num'] = df['user_id'].apply(lambda x: int(x[2:]))
    df = df.sort_values('num').drop(columns='num').reset_index(drop=True)

    return df


In [56]:
ratings_dates.head(3)

,userId,movieId,rating,date
0,1,1,4.0,2000-07-30 18:45:03
1,1,3,4.0,2000-07-30 18:20:47
2,1,6,4.0,2000-07-30 18:37:04


In [57]:
movies.head(3)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance


In [58]:
joined_false = ratings_dates.join(movies, rsuffix='_right',how='left')

In [59]:
joined_false

,userId,movieId,rating,date,movieId_right,title,genres
0,1,1,4.0,2000-07-30 18:45:03,1.0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,2000-07-30 18:20:47,2.0,Jumanji (1995),Adventure|Children|Fantasy
2,1,6,4.0,2000-07-30 18:37:04,3.0,Grumpier Old Men (1995),Comedy|Romance
3,1,47,5.0,2000-07-30 19:03:35,4.0,Waiting to Exhale (1995),Comedy|Drama|Romance
4,1,50,5.0,2000-07-30 18:48:51,5.0,Father of the Bride Part II (1995),Comedy
...,...,...,...,...,...,...,...
100831,610,166534,4.0,2017-05-03 21:53:22,NaN,NaN,NaN
100832,610,168248,5.0,2017-05-03 22:21:31,NaN,NaN,NaN
100833,610,168250,5.0,2017-05-08 19:50:47,NaN,NaN,NaN
100834,610,168252,5.0,2017-05-03 21:19:12,NaN,NaN,NaN


In [60]:
joined = ratings_dates.join(movies.set_index('movieId'), on='movieId',how='left')

In [61]:
joined

,userId,movieId,rating,date,title,genres
0,1,1,4.0,2000-07-30 18:45:03,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,2000-07-30 18:20:47,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,2000-07-30 18:37:04,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,2000-07-30 19:03:35,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,2000-07-30 18:48:51,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
...,...,...,...,...,...,...
100831,610,166534,4.0,2017-05-03 21:53:22,Split (2017),Drama|Horror|Thriller
100832,610,168248,5.0,2017-05-03 22:21:31,John Wick: Chapter Two (2017),Action|Crime|Thriller
100833,610,168250,5.0,2017-05-08 19:50:47,Get Out (2017),Horror
100834,610,168252,5.0,2017-05-03 21:19:12,Logan (2017),Action|Sci-Fi


In [62]:
merged = ratings_dates.merge(movies, on='movieId', how='left')

In [63]:
merged

,userId,movieId,rating,date,title,genres
0,1,1,4.0,2000-07-30 18:45:03,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,2000-07-30 18:20:47,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,2000-07-30 18:37:04,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,2000-07-30 19:03:35,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,2000-07-30 18:48:51,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
...,...,...,...,...,...,...
100831,610,166534,4.0,2017-05-03 21:53:22,Split (2017),Drama|Horror|Thriller
100832,610,168248,5.0,2017-05-03 22:21:31,John Wick: Chapter Two (2017),Action|Crime|Thriller
100833,610,168250,5.0,2017-05-08 19:50:47,Get Out (2017),Horror
100834,610,168252,5.0,2017-05-03 21:19:12,Logan (2017),Action|Sci-Fi


In [64]:
# merged.to_csv('merged.csv', index=False)

In [65]:
print('Число строк в таблице ratings_dates: ', ratings_dates.shape[0])
print('Число строк в таблице merged: ', merged.shape[0])
print(ratings_dates.shape[0] == merged.shape[0])

Число строк в таблице ratings_dates:  100836
Число строк в таблице merged:  100836
True


In [66]:
data_1 = pd.DataFrame({'Value': [100, 45, 80],
                       'Group': [1, 4, 5]},
                      index = ['I0', 'I1', 'I2']
                     )

data_2 = pd.DataFrame({'Company': ['Google', 'Amazon', 'Facebook'],
                       'Add': ['S0', 'S1', 'S7']},
                      index = ['I0', 'I1', 'I3']
                     )

In [67]:
my_conc = pd.concat([data_1, data_2], axis=1)

In [68]:
my_conc

,Value,Group,Company,Add
I0,100.0,1.0,Google,S0
I1,45.0,4.0,Amazon,S1
I2,80.0,5.0,NaN,NaN
I3,NaN,NaN,Facebook,S7


In [69]:
data_1.join(data_2, how='inner')

,Value,Group,Company,Add
I0,100,1,Google,S0
I1,45,4,Amazon,S1


In [70]:
a = pd.DataFrame({'A': ['a', 'b', 'c'], 'B': [103, 214, 124], 'C': [1, 4, 2]})
b = pd.DataFrame({'V': ['d', 'b', 'c'], 'U': [1393.7, 9382.2, 1904.5], 'C': [1, 3, 2]})

In [71]:
items_df = pd.DataFrame({
    'item_id': [417283, 849734, 132223, 573943, 19475, 3294095, 382043, 302948, 100132, 312394],
    'vendor': ['Samsung', 'LG', 'Apple', 'Apple', 'LG', 'Apple', 'Samsung', 'Samsung', 'LG', 'ZTE'],
    'stock_count': [54, 33, 122, 18, 102, 43, 77, 143, 60, 19]
})

purchase_df = pd.DataFrame({
    'purchase_id': [101, 101, 101, 112, 121, 145, 145, 145, 145, 221],
    'item_id': [417283, 849734, 132223, 573943, 19475, 3294095, 382043, 302948, 103845, 100132],
    'price': [13900, 5330, 38200, 49990, 9890, 33000, 67500, 34500, 89900, 11400]
})

In [72]:
display(items_df, purchase_df)

,item_id,vendor,stock_count
0,417283,Samsung,54
1,849734,LG,33
2,132223,Apple,122
3,573943,Apple,18
4,19475,LG,102
5,3294095,Apple,43
6,382043,Samsung,77
7,302948,Samsung,143
8,100132,LG,60
9,312394,ZTE,19


,purchase_id,item_id,price
0,101,417283,13900
1,101,849734,5330
2,101,132223,38200
3,112,573943,49990
4,121,19475,9890
5,145,3294095,33000
6,145,382043,67500
7,145,302948,34500
8,145,103845,89900
9,221,100132,11400


In [73]:
merged_2 = items_df.merge(purchase_df, how='inner')

In [74]:
income = (merged_2['price'] * merged_2['stock_count']).sum()

In [75]:
melb_df.groupby(by='Regionname', as_index=False, observed=False)['Distance'].min().sort_values(by='Distance', ascending=False, ignore_index=True)

,Regionname,Distance
0,Western Victoria,29.8
1,Eastern Victoria,25.2
2,Northern Victoria,21.8
3,South-Eastern Metropolitan,14.7
4,Eastern Metropolitan,7.8
5,Western Metropolitan,4.3
6,Southern Metropolitan,0.7
7,Northern Metropolitan,0.0


In [76]:
melb_df.groupby('MonthSale')['Price'].agg(['count', 'mean', 'median', 'max', 'min']).sort_values(by='count', ascending=False)

/var/folders/p3/8xj5n0vd5y132hpwjr8n1jw80000gn/T/ipykernel_2447/2632848814.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  melb_df.groupby('MonthSale')['Price'].agg(['count', 'mean', 'median', 'max', 'min']).sort_values(by='count', ascending=False)


,count,mean,median,max,min
MonthSale,,,,,
8,1850,1.056371e+06,892000.0,6500000.0,160000.0
7,1835,9.314698e+05,800000.0,9000000.0,190000.0
5,1644,1.097807e+06,905000.0,8000000.0,145000.0
6,1469,1.068981e+06,900000.0,7650000.0,222000.0
3,1408,1.146762e+06,945000.0,5600000.0,85000.0
4,1246,1.050479e+06,905500.0,5500000.0,145000.0
9,1188,1.126349e+06,980000.0,6400000.0,170000.0
10,854,1.135970e+06,950000.0,6250000.0,250000.0
11,750,1.142503e+06,950000.0,5050000.0,240000.0


In [77]:
reg_sellers = melb_df.groupby('Regionname',observed=False)['SellerG'].agg(['nunique', set])
reg_sellers

,nunique,set
Regionname,,
Eastern Metropolitan,26,"{McGrath, Buckingham, Buxton, Kay, RT, Fletche..."
Eastern Victoria,11,"{C21, McGrath, Harcourts, Eview, hockingstuart..."
Northern Metropolitan,40,"{McGrath, Buckingham, Kay, Village, Sweeney, R..."
Northern Victoria,11,"{LITTLE, McGrath, Buckingham, hockingstuart, M..."
South-Eastern Metropolitan,25,"{Buxton, McGrath, Greg, Fletchers, C21, Eview,..."
Southern Metropolitan,38,"{Buxton, McGrath, Buckingham, Kay, RT, Raine, ..."
Western Metropolitan,34,"{McGrath, Village, Sweeney, RT, Raine, Greg, B..."
Western Victoria,6,"{hockingstuart, other, HAR, Raine, Ray, YPA}"


In [78]:
melb_df.pivot_table(values='Price', index='Rooms', )

,Price
Rooms,
1,4.338245e+05
2,7.750812e+05
3,1.076081e+06
4,1.445282e+06
5,1.870260e+06
6,1.849366e+06
7,1.920700e+06
8,1.602750e+06
10,9.000000e+05


In [79]:
import re
result = re.findall(r'\w+', 'AV is largest Analytics community of India')
print(result)

['AV', 'is', 'largest', 'Analytics', 'community', 'of', 'India']


In [80]:
a1 = str(reg_sellers['set']['Eastern Metropolitan'])
a2 = re.findall(r'\w+', a1)
a1

"{'McGrath', 'Buckingham', 'Buxton', 'Kay', 'RT', 'Fletchers', 'C21', 'hockingstuart', 'Purplebricks', 'Jellis', 'Woodards', 'Ray', 'HAR', 'Barry', 'Nelson', 'Philip', 'RW', 'Love', 'other', 'Noel', 'Gary', 'Miles', 'Stockdale', 'Harcourts', 'Marshall', 'Biggin'}"

In [81]:
#библиотека для регулярных выражений
'''import re 
def get_year_release(arg):
    #находим все слова по шаблону "(DDDD)"
    candidates = re.findall(r'\(\d{4}\)', arg) 
    # проверяем число вхождений
    if len(candidates) > 0:
        #если число вхождений больше 0,
	#очищаем строку от знаков "(" и ")"
        year = candidates[0].replace('(', '')
        year = year.replace(')', '')
        return int(year)
    else:
        #если год не указан, возвращаем None
        return None'''

<>:5: SyntaxWarning: invalid escape sequence '\('
<>:5: SyntaxWarning: invalid escape sequence '\('
/var/folders/p3/8xj5n0vd5y132hpwjr8n1jw80000gn/T/ipykernel_2447/1873363390.py:5: SyntaxWarning: invalid escape sequence '\('
  candidates = re.findall(r'\(\d{4}\)', arg)


'import re \ndef get_year_release(arg):\n    #находим все слова по шаблону "(DDDD)"\n    candidates = re.findall(r\'\\(\\d{4}\\)\', arg) \n    # проверяем число вхождений\n    if len(candidates) > 0:\n        #если число вхождений больше 0,\n\t#очищаем строку от знаков "(" и ")"\n        year = candidates[0].replace(\'(\', \'\')\n        year = year.replace(\')\', \'\')\n        return int(year)\n    else:\n        #если год не указан, возвращаем None\n        return None'

In [82]:
import re
def get_year_release(arg):
    m = re.search(r"\((\d{4})\)", arg) if isinstance(arg, str) else None
    return int(m.group(1)) if m else pd.NA   # важно: pd.NA, а не None

In [83]:
merged.head(2)

,userId,movieId,rating,date,title,genres
0,1,1,4.0,2000-07-30 18:45:03,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,2000-07-30 18:20:47,Grumpier Old Men (1995),Comedy|Romance


In [84]:
merged['year_release'] = merged['title'].apply(get_year_release)

In [85]:
merged['year_release']

0         1995
1         1995
2         1995
3         1995
4         1995
          ... 
100831    2017
100832    2017
100833    2017
100834    2017
100835    2017
Name: year_release, Length: 100836, dtype: object

In [86]:
merged.head(2)

,userId,movieId,rating,date,title,genres,year_release
0,1,1,4.0,2000-07-30 18:45:03,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995
1,1,3,4.0,2000-07-30 18:20:47,Grumpier Old Men (1995),Comedy|Romance,1995


In [87]:
merged[merged['year_release'] == 1999].groupby('title', observed=False)['rating'].mean().sort_values(ascending=True)

title
Bloodsport: The Dark Kumite (1999)                  0.5
Trippin' (1999)                                     1.0
Chill Factor (1999)                                 1.0
From Dusk Till Dawn 2: Texas Blood Money (1999)     1.0
Simon Sez (1999)                                    1.0
                                                   ... 
Mickey's Once Upon a Christmas (1999)               5.0
On the Ropes (1999)                                 5.0
Trailer Park Boys (1999)                            5.0
Five Senses, The (1999)                             5.0
George Carlin: You Are All Diseased (1999)          5.0
Name: rating, Length: 261, dtype: float64

In [88]:
merged[merged['year_release'] == 2010].groupby('genres', observed=False)['rating'].mean().sort_values(ascending=True).round()

genres
Action|Sci-Fi                        1.0
Action|Adventure|Horror              2.0
Action|Drama|Fantasy                 2.0
Crime|Romance                        2.0
Adventure|Comedy|Fantasy             2.0
                                    ... 
Crime                                5.0
Adventure|Children|Comedy|Mystery    5.0
Animation|Children|Mystery           5.0
Animation|Drama|Fantasy|Mystery      5.0
Comedy|Musical                       5.0
Name: rating, Length: 119, dtype: float64

Какой пользователь (userId) посмотрел наибольшее количество различных (уникальных) комбинаций жанров (genres) фильмов? В качестве ответа запишите идентификатор этого пользователя.

In [97]:
an = merged.groupby('genres', observed=False)#['userId']

In [99]:
display(an)